# Small Dataset Tuning

小規模 HDF5 を使って、Jupyter 上で設定を変えながら学習と診断を行う notebook です。
この notebook では隠れたラッパー関数に処理を投げず、実際に使う `train_model`、入力分布、feature group ablation の呼び出しをセル内に直接書きます。

主に触る場所は `DATASET_KEY` と `train_kwargs` です。`flat200` と `flat500` は動作確認、`flat2000` は軽い比較、`flat20000` と `flat50000` は統計を見たい実験に使います。

## 1. 初期化

リポジトリを `sys.path` に入れ、学習本体と解析関数を import します。Jupyter の current directory がリポジトリでない場合だけ、既定の repo path にフォールバックします。

In [ ]:
from pathlib import Path
import inspect
import json
import sys

repo = Path.cwd()
if not (repo / "src" / "talesd_gnn_reconstruction").exists():
    repo = Path("/ceph/sharedfs/work/SATORI/ikomae/src/talesd_gnn_reconstruction")
sys.path.insert(0, str(repo / "src"))

from talesd_gnn_reconstruction.feature_analysis import (
    save_feature_group_importance,
    save_input_distributions,
)
from talesd_gnn_reconstruction.train import train_model

print(f"repo: {repo}")

## 2. 実行されるコードを確認する

下のセルは、この notebook から直接呼ぶ関数の実装を表示します。学習そのものは `train_model` が実行し、入力分布は `save_input_distributions`、特徴量 group の寄与評価は `save_feature_group_importance` が実行します。

In [ ]:
for obj in (train_model, save_input_distributions, save_feature_group_importance):
    print("=" * 100)
    print(f"{obj.__module__}.{obj.__name__}")
    print("=" * 100)
    print(inspect.getsource(obj))

## 3. データセットを選ぶ

`DATASET_KEY` を変えるだけで、使う HDF5 を切り替えます。ここでは HDF5 shard を持つディレクトリを指定します。存在しない場合は、この kernel からその path が見えていないという意味です。

In [ ]:
GRAPH_SETS = {
    "flat200": "/dicos_ui_home/ikomae/work/gnn/graphs/flat200",
    "flat500": "/dicos_ui_home/ikomae/work/gnn/graphs/flat500",
    "flat2000": "/dicos_ui_home/ikomae/work/gnn/graphs/flat2000",
    "flat20000": "/dicos_ui_home/ikomae/work/gnn/graphs/flat20000",
    "flat50000": "/dicos_ui_home/ikomae/work/gnn/graphs/flat50000",
}

DATASET_KEY = "flat2000"
graph_dir = Path(GRAPH_SETS[DATASET_KEY]).expanduser()
graph_files = sorted(graph_dir.glob("*.h5")) if graph_dir.exists() else []

print(f"dataset: {DATASET_KEY}")
print(f"graph_dir: {graph_dir}")
print(f"h5 shards: {len(graph_files)}")
for path in graph_files[:10]:
    print(f"  {path.name}")
if not graph_files:
    print("この kernel から graph shard が見えていません。DiCOS App の実行環境と path を確認してください。")

## 4. 入力パラメーター分布を描く

学習前に、ノード特徴量、エッジ特徴量、パルス特徴量、波形値、target の分布を確認します。全件を読むと重い場合があるため、まずは `max_graphs` を制限します。出力は PDF と JSON です。

In [ ]:
distribution_dir = Path("/dicos_ui_home/ikomae/work/gnn/outputs/talesd_gnn_reconstruction/notebook_diagnostics") / DATASET_KEY / "input_distributions"

RUN_INPUT_DISTRIBUTIONS = False
if RUN_INPUT_DISTRIBUTIONS:
    distribution_result = save_input_distributions(
        graph_dir,
        distribution_dir,
        max_graphs=100000,
        max_values_per_feature=200000,
        seed=12345,
        show_progress=True,
    )
    print(json.dumps(distribution_result, indent=2)[:8000])
else:
    print("RUN_INPUT_DISTRIBUTIONS=True にすると分布図を作成します。")
    print(f"output: {distribution_dir}")

## 5. 学習設定

`train_kwargs` が `train_model` にそのまま渡る設定です。script 実行と対応させるため、標準設定は `physics + cnn-gru + quality` にしています。`save_diagnostics=True` の場合、epoch ごとに loss curve が更新され、validation loss の best が更新されたときに軽量診断図も更新されます。

In [ ]:
run_name = f"notebook_{DATASET_KEY}_reco_quality"
run_dir = Path("/dicos_ui_home/ikomae/work/gnn/outputs/talesd_gnn_reconstruction/runs") / run_name
checkpoint = run_dir / "checkpoints" / f"{run_name}.pt"

train_kwargs = dict(
    graphs_path=str(graph_dir),
    output_path=str(checkpoint),
    epochs=16,
    batch_size=256,
    learning_rate=3e-4,
    weight_decay=1e-4,
    hidden_dim=192,
    num_layers=5,
    dropout=0.05,
    lr_scheduler="reduce-on-plateau",
    lr_factor=0.5,
    lr_patience=2,
    early_stopping_patience=12,
    early_stopping_min_epochs=32,
    model_architecture="physics",
    readout_heads=4,
    classification_arch="enhanced",
    detector_embedding_dim=0,
    waveform_encoder="cnn-gru",
    waveform_embedding_dim=64,
    waveform_transformer_heads=4,
    waveform_transformer_layers=1,
    loss_mode="physics",
    energy_loss_weight=1.2,
    core_loss_weight=1.0,
    direction_loss_weight=1.4,
    core_loss_scale_km=0.12,
    angular_loss_scale_deg=1.0,
    val_fraction=0.05,
    test_fraction=0.10,
    split_mode="source-stratified",
    seed=12345,
    device="auto",
    sample_cache_size=0,
    max_graphs=None,
    particle_filter="all",
    pin_memory=None,
    num_workers=4,
    preprocess_workers=8,
    prefetch_factor=2,
    persistent_workers=False,
    collate_backend="cpp",
    collate_threads=1,
    training_task="reconstruction",
    mass_classification=False,
    quality_prediction=True,
    quality_loss_weight=0.2,
    error_prediction=False,
    nll_loss_weight=0.0,
    show_progress=True,
    save_diagnostics=True,
    update_learning_curve_each_epoch=True,
    best_diagnostics=True,
    best_diagnostic_max_graphs=20000,
    diagnostic_energy_bin_width=0.1,
    diagnostic_min_bin_count=1000,
)

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in train_kwargs.items()}, indent=2)[:12000])

## 6. 学習を実行する

`RUN_TRAINING=True` にすると、上の `train_kwargs` で学習を開始します。ここでは `train_model(**train_kwargs)` を直接呼ぶため、実際に使う引数が notebook 上で見えます。

`LIVE_LOSS_PLOT=True` の場合は、各 epoch が終わるたびに notebook の出力セル内で loss curve を更新します。batch ごとの描画は重いため行いません。

In [ ]:
RUN_TRAINING = False
LIVE_LOSS_PLOT = True

def live_loss_callback(history, output_path):
    from IPython.display import clear_output, display
    import matplotlib.pyplot as plt

    epochs = [row["epoch"] for row in history]
    train_loss = [row.get("train_loss") for row in history]
    val_loss = [row.get("val_loss") for row in history]
    finite_val = [(i, value) for i, value in enumerate(val_loss) if value is not None]
    best_index = min(finite_val, key=lambda item: item[1])[0] if finite_val else None

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(epochs, train_loss, marker="o", linewidth=1.4, label="train loss")
    ax.plot(epochs, val_loss, marker="o", linewidth=1.4, label="validation loss")
    if best_index is not None:
        ax.axvline(epochs[best_index], color="0.35", linestyle="--", linewidth=1.0, label=f"best epoch {epochs[best_index]}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title("training progress")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(frameon=False)
    fig.tight_layout()

    clear_output(wait=True)
    display(fig)
    plt.close(fig)
    latest = history[-1]
    best_text = "none" if best_index is None else f"epoch={epochs[best_index]} val_loss={val_loss[best_index]:.6g}"
    print(f"latest epoch={latest['epoch']} train_loss={latest.get('train_loss'):.6g} val_loss={latest.get('val_loss'):.6g}")
    print(f"best: {best_text}")
    print(f"learning_curve_pdf: {Path(str(output_path) + '.diagnostics') / 'learning_curve.pdf'}")

if RUN_TRAINING:
    train_kwargs_live = dict(train_kwargs)
    if LIVE_LOSS_PLOT:
        train_kwargs_live["epoch_callback"] = live_loss_callback
    result = train_model(**train_kwargs_live)
    print(json.dumps(result, indent=2)[:8000])
else:
    print("RUN_TRAINING=True にすると train_model(**train_kwargs) を実行します。")
    print(f"checkpoint: {checkpoint}")

## 7. 特徴量 group の寄与を評価する

学習済み checkpoint に対して、特徴量 group を学習データ平均で置き換え、validation 指標がどれだけ悪化するかを測ります。これは再学習を行わない post-hoc ablation です。`flat50000` でも再学習を何十回も回す必要はありません。

In [ ]:
importance_dir = run_dir / "feature_importance"

RUN_FEATURE_IMPORTANCE = False
if RUN_FEATURE_IMPORTANCE:
    importance_result = save_feature_group_importance(
        graph_dir,
        checkpoint,
        importance_dir,
        split="validation",
        max_graphs=50000,
        batch_size=train_kwargs["batch_size"],
        device=train_kwargs["device"],
        seed=train_kwargs["seed"],
        show_progress=True,
    )
    print(json.dumps(importance_result, indent=2)[:12000])
else:
    print("RUN_FEATURE_IMPORTANCE=True にすると feature group ablation を実行します。")
    print(f"output: {importance_dir}")